#### Group Fairness 모델
- FairGNN (Adversarial Learning): 대표
    - FairVGNN (Adversarial Learning)

- EDITS (Edge Rewiring): 대표
    - UGE (Edge Rewiring) : 제외
    - FairEdit (Edge Rewiring)
    - GEAR (Edge Rewiring)

- FairWalk (Rebalancing): 대표
    - CrossWalk (Rebalancing)

- NIFTY (Optimization with Regularization)

- FairGB

In [2]:
import pandas as pd

### Avg.

In [148]:
df1 = pd.read_csv('./outputs/exp_fairgate_gcn.csv')
df1 = df1[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df2 = pd.read_csv('./outputs/compare/exp_baselines.csv')
df2 = df2[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df = pd.concat([df1, df2])
df_rank_v1 = df[df["model"] != "GCN"].copy()  # baseline 제외

df_rank = df_rank_v1[df_rank_v1['dataset'] != 'recidivism'].copy() # recidivism 제외시
# df_rank = df_rank_v1.copy() # 제외 아닐 시

# dataset별 rank 계산
df_rank["rank_Acc"] = df_rank.groupby("dataset")["acc_mean"].rank(ascending=False, method="average")
df_rank["rank_AUC"] = df_rank.groupby("dataset")["roc_auc_mean"].rank(ascending=False, method="average")
df_rank["rank_DP"]  = df_rank.groupby("dataset")["dp_mean"].rank(ascending=True,  method="average")
df_rank["rank_EO"]  = df_rank.groupby("dataset")["eo_mean"].rank(ascending=True,  method="average")

# accuracy / fairness 평균 rank
df_rank["rank_acc_avg"] = df_rank[["rank_Acc", "rank_AUC"]].mean(axis=1)
df_rank["rank_fair_avg"] = df_rank[["rank_DP", "rank_EO"]].mean(axis=1)

# 전체 dataset에 대한 모델별 평균 rank
model_order = ['FairGNN', 'FairVGNN', 
               'FairEdit', 'EDITS', 
               'FairWalk', 'CrossWalk', 
               'FairGB', 'NIFTY', 
               'FairGT', 'FairGate']

summary = (
    df_rank.groupby("model")[["rank_Acc", "rank_AUC", "rank_DP", "rank_EO",
                              "rank_acc_avg", "rank_fair_avg"]]
    .mean()
)

summary = summary.reindex(model_order)

print(summary.round(4))

           rank_Acc  rank_AUC  rank_DP  rank_EO  rank_acc_avg  rank_fair_avg
model                                                                       
FairGNN       2.125     5.250    6.750    6.625        3.6875         6.6875
FairVGNN      5.000     3.125    4.875    4.875        4.0625         4.8750
FairEdit      4.500     6.250    6.000    6.750        5.3750         6.3750
EDITS         9.500     4.500    9.500    7.000        7.0000         8.2500
FairWalk      5.875     7.250    3.375    4.625        6.5625         4.0000
CrossWalk     5.875     7.125    4.250    4.750        6.5000         4.5000
FairGB        4.750     1.750    6.500    4.250        3.2500         5.3750
NIFTY         5.875     6.250    5.875    5.875        6.0625         5.8750
FairGT        5.625     3.375    3.750    3.375        4.5000         3.5625
FairGate      1.625     1.500    1.500    1.625        1.5625         1.5625


In [5]:
df1 = pd.read_csv('./outputs/exp_fairgate_gcn.csv')
df1 = df1[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df2 = pd.read_csv('./outputs/compare/exp_baselines.csv')
df2 = df2[['model', 'dataset', 'acc_mean', 'roc_auc_mean', 'dp_mean', 'eo_mean']]

df = pd.concat([df1, df2], ignore_index=True)


df_rank_v1 = df[df["model"] != "GCN"].copy()   # baseline 제외
# df_rank = df_rank_v1[df_rank_v1['dataset'] != 'recidivism'].copy() # recidivism 제외
df_rank = df_rank_v1.copy()


df_rank["rank_Acc"] = df_rank.groupby("dataset")["acc_mean"].rank(
    ascending=False, method="average"
)
df_rank["rank_AUC"] = df_rank.groupby("dataset")["roc_auc_mean"].rank(
    ascending=False, method="average"
)
df_rank["rank_DP"] = df_rank.groupby("dataset")["dp_mean"].rank(
    ascending=True, method="average"
)
df_rank["rank_EO"] = df_rank.groupby("dataset")["eo_mean"].rank(
    ascending=True, method="average"
)

# metric-group average rank
df_rank["rank_acc_avg"] = df_rank[["rank_Acc", "rank_AUC"]].mean(axis=1)
df_rank["rank_fair_avg"] = df_rank[["rank_DP", "rank_EO"]].mean(axis=1)


df_tmp = (
    df_rank.groupby(["model", "dataset"], as_index=False)[
        ["rank_Acc", "rank_AUC", "rank_DP", "rank_EO",
         "rank_acc_avg", "rank_fair_avg"]
    ]
    .mean()
)

summary = (
    df_tmp.groupby("model")[
        ["rank_Acc", "rank_AUC", "rank_DP", "rank_EO",
         "rank_acc_avg", "rank_fair_avg"]
    ]
    .mean()
)

# dataset coverage count
summary["num_datasets"] = df_tmp.groupby("model")["dataset"].nunique()

model_order = [
    'FairGNN', 'FairVGNN',
    'FairEdit', 'EDITS',
    'FairWalk', 'CrossWalk',
    'FairGB', 'NIFTY',
    'FairGT', 'FairGate'
]

summary = summary.reindex(model_order)

# =========================
# 6. Print
# =========================
print(summary.round(2))

           rank_Acc  rank_AUC  rank_DP  rank_EO  rank_acc_avg  rank_fair_avg  \
model                                                                          
FairGNN        2.11      5.44     7.56     7.22          3.78           7.39   
FairVGNN       5.56      3.67     5.11     5.67          4.61           5.39   
FairEdit       5.20      6.60     5.80     6.80          5.90           6.30   
EDITS          8.00      5.33     8.67     5.67          6.67           7.17   
FairWalk       6.78      8.00     3.44     4.67          7.39           4.06   
CrossWalk      6.44      7.78     4.44     4.94          7.11           4.69   
FairGB         3.56      2.00     5.11     3.83          2.78           4.47   
NIFTY          5.89      6.56     6.56     6.22          6.22           6.39   
FairGT         5.89      3.67     3.56     4.11          4.78           3.83   
FairGate       2.56      1.78     2.44     2.00          2.17           2.22   

           num_datasets  
model        

### ours

In [3]:
file_name ='exp_fairgate_gcn'

datasets = [
    'pokec_z', 'pokec_n', 
    'pokec_z_g', 'pokec_n_g',
    'credit', 'recidivism', 'income', 
    'german', 
    'nba'
             ]

df = pd.read_csv(f'./outputs/{file_name}.csv')
df = df[[
    'model', 'dataset',
    'acc_mean', 
    'acc_std', 
    'roc_auc_mean', 
    'roc_auc_std',
    'dp_mean', 
    'dp_std', 
    'eo_mean', 
    'eo_std', 
    # 'time_sec_mean', 'time_sec_std'
]]

df = df.copy()
df['dataset'] = pd.Categorical(
    df['dataset'],
    categories=datasets,
    ordered=True
)

df.sort_values('dataset')
# [df.dataset == 'german']

,model,dataset,acc_mean,acc_std,roc_auc_mean,roc_auc_std,dp_mean,dp_std,eo_mean,eo_std
0,FairGate,pokec_z,0.6754,0.0020,0.7393,0.0014,0.0071,0.0034,0.0072,0.0059
1,FairGate,pokec_n,0.6866,0.0047,0.7290,0.0030,0.0370,0.0119,0.0477,0.0076
2,FairGate,pokec_z_g,0.7000,0.0043,0.7613,0.0060,0.0278,0.0215,0.0159,0.0098
3,FairGate,pokec_n_g,0.6664,0.0077,0.7238,0.0017,0.0080,0.0081,0.0501,0.0165
8,FairGate,credit,0.7972,0.0030,0.7301,0.0051,0.0262,0.0047,0.0064,0.0039
5,FairGate,recidivism,0.8490,0.0029,0.8929,0.0008,0.0703,0.0054,0.0396,0.0036
7,FairGate,income,0.8106,0.0063,0.7682,0.0024,0.0332,0.0127,0.0170,0.0112
4,FairGate,german,0.6896,0.0122,0.6510,0.0275,0.0209,0.0270,0.0326,0.0483
6,FairGate,nba,0.7042,0.0378,0.7772,0.0053,0.0244,0.0231,0.0411,0.0375


In [3]:
file_name ='exp_fairgate_sc'

datasets = ['pokec_z', 'pokec_n', 'pokec_z_g', 'pokec_n_g',
             'credit', 'recidivism', 'income', 'german', 'nba']

df = pd.read_csv(f'./outputs/{file_name}.csv')
df = df[[
    'model', 'dataset',
    'acc_mean', 
    'acc_std', 
    'roc_auc_mean', 
    'roc_auc_std',
    'dp_mean', 
    'dp_std', 
    'eo_mean', 
    'eo_std', 
    # 'time_sec_mean', 'time_sec_std'
]]

df = df.copy()
df['dataset'] = pd.Categorical(
    df['dataset'],
    categories=datasets,
    ordered=True
)

df.sort_values('dataset')
# [df.dataset == 'credit']

,model,dataset,acc_mean,acc_std,roc_auc_mean,roc_auc_std,dp_mean,dp_std,eo_mean,eo_std
0,FairGate,pokec_z,0.6754,0.0015,0.7398,0.0021,0.0082,0.0060,0.0077,0.0050
1,FairGate,pokec_n,0.6882,0.0034,0.7298,0.0032,0.0332,0.0102,0.0454,0.0102
2,FairGate,pokec_z_g,0.6990,0.0069,0.7580,0.0049,0.0255,0.0139,0.0120,0.0090
3,FairGate,pokec_n_g,0.6694,0.0080,0.7240,0.0021,0.0089,0.0059,0.0502,0.0115
5,FairGate,credit,0.7947,0.0085,0.7292,0.0046,0.0348,0.0206,0.0148,0.0175
7,FairGate,recidivism,0.8497,0.0029,0.8928,0.0017,0.0702,0.0039,0.0402,0.0031
6,FairGate,income,0.8090,0.0063,0.7698,0.0024,0.0311,0.0136,0.0147,0.0114
4,FairGate,german,0.6916,0.0137,0.6505,0.0253,0.0211,0.0217,0.0267,0.0384
8,FairGate,nba,0.6958,0.0388,0.7768,0.0041,0.0238,0.0155,0.0388,0.0284


### baselines

In [ ]:
file_name ='exp_baselines'
df_v2 = pd.read_csv(f'./outputs/compare/{file_name}.csv')
df_v2 = df_v2[[
    'model', 'dataset',
    'acc_mean', 
    'acc_std', 
    'roc_auc_mean',
    'roc_auc_std',
    'dp_mean',
    'dp_std',
    'eo_mean', 
    'eo_std', 
    # 'time_sec_mean', 
    # 'time_sec_std'
]]

datasets = ['pokec_z', 'pokec_n', 'pokec_z_g', 'pokec_n_g',
             'credit', 'recidivism', 'income', 'german', 'nba']
model_order = ['FairGNN', 'FairVGNN', 'FairEdit', 'EDITS',
               'FairWalk', 'CrossWalk', 'FairGB', 'NIFTY', 'FairGT']

filtered_df = df_v2.copy()
filtered_df['model'] = pd.Categorical(
    filtered_df['model'],
    categories=model_order,
    ordered=True
)
filtered_df['dataset'] = pd.Categorical(
    filtered_df['dataset'],
    categories=datasets,
    ordered=True
)

filtered_df = filtered_df.sort_values(['model', 'dataset'])
# filtered_df[filtered_df.dataset == 'recidivism']
filtered_df

,model,dataset,acc_mean,acc_std,roc_auc_mean,roc_auc_std,dp_mean,dp_std,eo_mean,eo_std
0,FairGB,pokec_z,0.6679,0.0156,0.7231,0.0116,0.0676,0.0263,0.0427,0.0299
1,FairGB,pokec_n,0.6763,0.0045,0.7222,0.0009,0.0799,0.0145,0.0959,0.0136
2,FairGB,pokec_z_g,0.6785,0.0070,0.7351,0.0074,0.0141,0.0098,0.0260,0.0194
3,FairGB,pokec_n_g,0.6713,0.0044,0.7221,0.0038,0.0225,0.0161,0.0510,0.0301


In [28]:
file_name ='exp_baselines_sc'
df_v2 = pd.read_csv(f'./outputs/{file_name}.csv')
df_v2 = df_v2[[
    'model', 'dataset',
    'acc_mean', 
    # 'acc_std', 
    'roc_auc_mean',
    # 'roc_auc_std',
    'dp_mean',
    # 'dp_std',
    'eo_mean', 
    # 'eo_std', 
    # 'time_sec_mean', 
    # 'time_sec_std'
]]

datasets = ['pokec_z', 'pokec_n', 'pokec_z_g', 'pokec_n_g',
             'credit', 'recidivism', 'income', 'german', 'nba']
model_order = ['FairGNN', 'FairVGNN', 'FairEdit', 'EDITS',
               'FairWalk', 'CrossWalk', 'FairGB', 'NIFTY', 'FairGT']

filtered_df = df_v2.copy()
filtered_df['model'] = pd.Categorical(
    filtered_df['model'],
    categories=model_order,
    ordered=True
)
filtered_df['dataset'] = pd.Categorical(
    filtered_df['dataset'],
    categories=datasets,
    ordered=True
)

filtered_df = filtered_df.sort_values(['model', 'dataset'])
filtered_df[filtered_df.model == 'CrossWalk']
# filtered_df

,model,dataset,acc_mean,roc_auc_mean,dp_mean,eo_mean
64,CrossWalk,pokec_z,0.6018,0.6082,0.0277,0.0318
65,CrossWalk,pokec_n,0.6223,0.6190,0.0307,0.0297
66,CrossWalk,pokec_z_g,0.6018,0.6082,0.0411,0.0592
67,CrossWalk,pokec_n_g,0.6223,0.6190,0.0732,0.1031
69,CrossWalk,credit,0.7808,0.6112,0.0846,0.0660
71,CrossWalk,recidivism,0.8507,0.8248,0.0566,0.0307
70,CrossWalk,income,0.7929,0.6066,0.0663,0.1014
68,CrossWalk,german,0.6356,0.5268,0.0822,0.0770
72,CrossWalk,nba,0.5122,0.5095,0.0406,0.0868


### gcn

In [ ]:
file_name ='exp_gcn'
df_ab = pd.read_csv(f'./outputs/compare/{file_name}.csv')
# df_ab = df_ab[df_ab.dataset == 'recidivism']
df_ab = df_ab.drop(['stage_label', 'run', 'seed'], axis=1)

df_ab.groupby(['dataset', 'stage']).agg(['mean', 'std']).round(4)

acc         roc_auc              f1              dp  \
                    mean     std    mean     std    mean     std    mean   
dataset    stage                                                           
credit     A0     0.7468  0.0018  0.7423  0.0022  0.8283  0.0016  0.1436   
german     A0     0.6120  0.0049  0.6459  0.0082  0.6768  0.0063  0.4957   
income     A0     0.7994  0.0007  0.7868  0.0019  0.5302  0.0058  0.1841   
nba        A0     0.7371  0.0152  0.7803  0.0027  0.7652  0.0220  0.0396   
pokec_n    A0     0.6904  0.0028  0.7368  0.0015  0.6514  0.0042  0.1058   
pokec_n_g  A0     0.6719  0.0108  0.7282  0.0070  0.6637  0.0053  0.0136   
pokec_z    A0     0.6878  0.0024  0.7581  0.0012  0.7021  0.0031  0.0647   
pokec_z_g  A0     0.6907  0.0023  0.7566  0.0022  0.6940  0.0055  0.1070   
recidivism A0     0.8504  0.0038  0.8916  0.0012  0.7977  0.0039  0.0755   

                              eo         time_sec          
                     std    mean     std     mean     std  
dataset    stage                                           
credit     A0     0.0030  0.1200  0.0037    16.02  2.8700  
german     A0     0.0116  0.5334  0.0103     6.06  2.7646  
income     A0     0.0043  0.2586  0.0089    14.74  2.4172  
nba        A0     0.0118  0.0528  0.0254     9.42  1.5595  
pokec_n    A0     0.0081  0.1252  0.0165     7.34  0.2302  
pokec_n_g  A0     0.0091  0.0501  0.0148    13.20  3.5057  
pokec_z    A0     0.0101  0.0515  0.0097     8.34  0.5857  
pokec_z_g  A0     0.0088  0.0471  0.0066     8.70  0.7550  
recidivism A0     0.0030  0.0422  0.0034    10.80  2.6038